# 第109章 层次聚类与DBSCAN

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 24 / 34 步：深入客户分群与降维表达**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** K-Means客户分群实战  →  **本章任务：** 层次聚类与DBSCAN  →  **下一步：** PCA降维与可视化
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景

拿到一堆只有样本、没有标签的数据时，最头疼的问题往往是「这些点到底该分成几组、边界又在哪里」。


## 本章目标

学完本章，你将能够：

- **理解**：理解「层次聚类与DBSCAN」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「层次聚类与DBSCAN」的关键输出指标。
- **迁移**：能把「层次聚类与DBSCAN」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 核心概念

**背景引入**：拿到一堆只有样本、没有标签的数据时，最头疼的问题往往是「这些点到底该分成几组、边界又在哪里」。层次聚类和 DBSCAN 正是解决这类问题的常用工具——前者把数据看成能不断合并的一棵树，后者则按「密度」把挤在一起的样本抱成团，还能把散落的离群点单独标出来。学会之后，面对不规则形状的会员分组或人群行为数据，你也知道该用哪种聚法了。

- 层次聚类逐步合并最近簇（打个比方：像给一群人按“谁跟谁熟”排家谱，从最像的两两开始不断合并；而 DBSCAN 则像看“人多才抱团”，周围没人的点就归为散客。）
- single/complete/ward 定义不同簇间距离
- DBSCAN 使用 eps 与 min_samples 定义核心点
- 密度方法可发现非球形簇和噪声


## 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 数据与问题定义 | `.fit_transform()` | 先明确样本、特征、目标和验证方式，再训练模型。 | 未缩放就设置 eps |
| 模型、公式与诊断 | `pd.DataFrame()`、`.fit()`、`.sum()`、`.round()` | 把核心数学量映射到 sklearn 输出，并检查泛化表现。 | 把 DBSCAN 噪声强制归入普通簇 |


## 例 1｜数据与问题定义

先明确样本、特征、目标和验证方式，再训练模型。


<!-- math-foundation:chapter-109 -->
### 数学推导｜距离决定聚类结果

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜先计算逐特征差异。** $\Delta_j=x_j-y_j$。

**第 2 步｜选择如何汇总差异。** 欧氏距离对大差异平方后更敏感，曼哈顿距离把绝对差异直接相加。

**第 3 步｜看尺度为什么重要。** 若改用标准化坐标 $z_j=(x_j-\mu_j)/\sigma_j$，则欧氏距离变为

$$
d_z(x,y)=\sqrt{\sum_j\left(\frac{x_j-y_j}{\sigma_j}\right)^2}
$$

这等于让每一维按自身尺度参与比较，避免大单位变量天然主导距离。

**把上面的关系收束为本章计算式：**

$$
d_2(x,y)=\sqrt{\sum_j(x_j-y_j)^2},\qquad d_1(x,y)=\sum_j|x_j-y_j|
$$

**符号解释：** $d_2$ 是欧氏距离，$d_1$ 是曼哈顿距离。

**代码对应：** 聚类前标准化特征，并说明距离与 linkage/邻域参数的选择。

**使用边界：** 不同量纲会让大数值特征主导距离；聚类簇不是天然存在的真实类别。


In [ ]:
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler

X, y = make_moons(n_samples=500, noise=0.08, random_state=98)
Xs = StandardScaler().fit_transform(X)


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**：示例 1 用 `make_moons(n_samples=500, noise=0.08, random_state=98)` 造出了两个交错成月牙形的簇。现在请把噪声参数 `noise` 从 `0.08` 调大到 `0.30`，其余保持原样，再数一数样本量 `X_ex.shape[0]` 和类别数 `len(set(y_ex))`。噪声变大只会让两个月牙的「边界」更离散，但样本数和类别数理应不受影响——改动后运行下方脚手架，用自检核对你的理解。


In [ ]:
try:
    # 请在下方填写代码：把 make_moons 的噪声参数 noise 改成 0.30，其余参数保持不变。
    # 提示：noise 控制两个月牙簇的离散程度，示例 1 用的值是 0.08。
    noise_to_try = None  # 待填空：把 None 改成 0.30

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜模型、公式与诊断

把核心数学量映射到 sklearn 输出，并检查泛化表现。


In [ ]:
import pandas as pd
from sklearn.cluster import AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score

agg = AgglomerativeClustering(n_clusters=2, linkage="ward").fit(Xs)
db = DBSCAN(eps=0.25, min_samples=6).fit(Xs)
rows = [
    [
        "层次聚类",
        len(set(agg.labels_)),
        0,
        adjusted_rand_score(y, agg.labels_),
    ],
    [
        "DBSCAN",
        len(set(db.labels_) - {-1}),
        (db.labels_ == -1).sum(),
        adjusted_rand_score(y, db.labels_),
    ],
]
display(pd.DataFrame(rows, columns=["模型", "簇数", "噪声点", "ARI"]).round(3))


## 独立迁移练习

在不改变数据切分和指标的前提下，比较基线与一个模型设置。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # TODO: 在此粘贴或改写最接近的示例。
    # 记录：我改了什么？预期会发生什么？实际观察到什么？
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(
        {"修改": change_note, "预期": expected_change, "观察": observed_change}
    )

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 本章实训：模型与基线比较

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X = pd.DataFrame(
    {"visits": [1, 2, 3, 4, 5, 6], "discount": [0, 0, 1, 1, 1, 2]}
)
y = np.array([12, 15, 19, 23, 27, 31])
baseline = DummyRegressor(strategy="mean").fit(X, y)
model = LinearRegression().fit(X, y)
print("基线预测：", np.round(baseline.predict(X[:2]), 2))
print("模型预测：", np.round(model.predict(X[:2]), 2))
print("基线MAE：", round(mean_absolute_error(y, baseline.predict(X)), 2))
print("模型MAE：", round(mean_absolute_error(y, model.predict(X)), 2))


### 第一个结果怎么读

复杂模型之前先建立基线。只有在同一数据切分和同一指标下超过基线，模型才值得继续分析。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
X_changed = X.copy()
X_changed["visits"] = X_changed["visits"] + 1
changed_prediction = model.predict(X_changed)
print("原始前2个预测：", np.round(model.predict(X[:2]), 2))
print("访问次数+1后的预测：", np.round(changed_prediction[:2], 2))
print("预测变化：", np.round(changed_prediction[:2] - model.predict(X[:2]), 2))


### 第二个结果怎么读

只把一个特征整体加 1，观察预测变化。这个实验只能说明模型的预测响应，不能直接证明真实世界的因果关系。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：模型特征泄漏怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

data = pd.DataFrame(
    {
        "visits": [2, 4, 6],
        "duration_after_call": [30, 80, 120],
        "target": [0, 1, 1],
    }
)
forbidden = {"target", "duration_after_call"}
features = [column for column in data.columns if column not in forbidden]
print("禁止使用：", sorted(forbidden))
print("安全特征：", features)
print("原因：特征必须在预测时点已经可获得。")


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

如果一个字段在结果发生之后才产生，它即使与目标高度相关，也不能作为预测特征。先定义预测时点，再列可用字段。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 未缩放就设置 eps
- 把 DBSCAN 噪声强制归入普通簇
- 只用轮廓系数评价非凸簇
- 在大数据上忽略层次聚类的内存成本


## 练习与作业

1. 修改一个关键参数并重新运行
2. 记录指标变化并解释原因
3. 检查结论是否依赖测试集或隐藏泄漏

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 109.11 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“修改一个关键参数并重新运行”。
2. **独立完成**：不复制示例代码，完成“记录指标变化并解释原因”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“检查结论是否依赖测试集或隐藏泄漏”，用一两句话说明你修改了什么。

### 109.11.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 109.11.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


## 小结

比较层次聚类和 DBSCAN，理解连接方式、密度邻域和噪声点。


### 你已经掌握

- 训练 AgglomerativeClustering
- 理解 linkage
- 训练 DBSCAN
- 识别噪声标签 -1


### 需要注意

- 未缩放就设置 eps
- 把 DBSCAN 噪声强制归入普通簇
- 只用轮廓系数评价非凸簇
- 在大数据上忽略层次聚类的内存成本


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
# 完整答案：噪声参数只影响点的离散程度，不影响样本数类别数。
X_ex, y_ex = make_moons(n_samples=500, noise=0.30, random_state=98)
print("样本数 =", X_ex.shape[0])
print("类别数 =", len(set(y_ex)))


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
practice = []
for eps in [0.15, 0.25, 0.4]:
    labels = DBSCAN(eps=eps, min_samples=6).fit_predict(Xs)
    practice.append([eps, len(set(labels) - {-1}), (labels == -1).sum()])
practice_result = pd.DataFrame(practice, columns=["eps", "clusters", "noise"])
display(practice_result)
